# 03 — Merge & Prepare ML Dataset

**Goal:** Load the cleaned historical panel, define features/targets, split into train/test.  
**Reads:** `data/processed/historical_panel.csv`  
**Outputs:** `data/processed/train.csv`, `data/processed/test.csv`

Features (5): `gdp_per_capita`, `hdi`, `control_of_corruption`, `employment_agriculture`, `gini`  
Targets (3): `poverty_3`, `poverty_8_30`, `poverty_10`  
Primary target: `poverty_3`

> `poverty_4_20` excluded — source file contains poverty gap, not headcount ratio.

## Imports and paths

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from pathlib import Path

DATA_PROCESSED = Path("../data/processed")
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

FEATURES = ["gdp_per_capita", "hdi", "control_of_corruption", "employment_agriculture", "gini"]
TARGETS = ["poverty_3", "poverty_8_30", "poverty_10"]  # poverty_4_20 excluded (poverty gap data)
PRIMARY_TARGET = "poverty_3"

print(f"Features ({len(FEATURES)}): {FEATURES}")
print(f"Targets  ({len(TARGETS)}): {TARGETS}")
print(f"Primary target: {PRIMARY_TARGET}")

Features (5): ['gdp_per_capita', 'hdi', 'control_of_corruption', 'employment_agriculture', 'gini']
Targets  (3): ['poverty_3', 'poverty_8_30', 'poverty_10']
Primary target: poverty_3


## Load historical panel

In [2]:
panel = pd.read_csv(DATA_PROCESSED / "historical_panel.csv")
print(f"Raw panel: {panel.shape}")
print(f"Countries: {panel['country_code'].nunique()}, Years: {panel['year'].min()}–{panel['year'].max()}")
panel.head()

Raw panel: (6076, 14)
Countries: 217, Years: 1996–2023


,country_name,country_code,year,gdp_per_capita,hdi,control_of_corruption,employment_agriculture,gini,gini_observed,poverty_3,poverty_8_30,poverty_10,imputation_pct,target_missing_pct
0,Aruba,ABW,1996,17615.984713,0.6485,-0.260567,26.144256,39.750,False,NaN,NaN,NaN,0.8,1.0
1,Aruba,ABW,1997,18963.341744,0.6450,-0.251114,25.729159,39.750,False,NaN,NaN,NaN,0.8,1.0
2,Aruba,ABW,1998,20009.938247,0.6515,-0.259514,25.015691,39.000,False,NaN,NaN,NaN,0.8,1.0
3,Aruba,ABW,1999,20109.916474,0.6580,-0.229512,23.876970,38.675,False,NaN,NaN,NaN,0.8,1.0
4,Aruba,ABW,2000,21259.759356,0.6660,-0.222040,23.290873,39.800,False,NaN,NaN,NaN,0.8,1.0


In [3]:
# Quick look at missing data before filtering
print("% NaN per column:")
print((panel[FEATURES + TARGETS].isna().mean() * 100).round(2))

% NaN per column:
gdp_per_capita             0.00
hdi                        0.00
control_of_corruption      0.00
employment_agriculture     0.00
gini                       0.00
poverty_3                 67.10
poverty_8_30              69.31
poverty_10                69.31
dtype: float64


## Filter rows

- Drop rows where **all** target columns are NaN (no poverty data at all)
- Drop rows where **all** feature columns are NaN

In [4]:
n_before = len(panel)

# Drop rows where the PRIMARY target is NaN — we need real poverty observations to train on
panel = panel.dropna(subset=[PRIMARY_TARGET])
print(f"After dropping NaN {PRIMARY_TARGET}: {len(panel)} (removed {n_before - len(panel)})")

# Drop rows with ALL features NaN
n_before2 = len(panel)
panel = panel.dropna(subset=FEATURES, how="all")
print(f"After dropping all-NaN features: {len(panel)} (removed {n_before2 - len(panel)})")

print(f"\nFinal dataset: {panel.shape}")
print(f"Countries: {panel['country_code'].nunique()}")

After dropping NaN poverty_3: 1999 (removed 4077)
After dropping all-NaN features: 1999 (removed 0)

Final dataset: (1999, 14)
Countries: 170


### Check primary target coverage

In [5]:
for t in TARGETS:
    n_valid = panel[t].notna().sum()
    print(f"{t}: {n_valid} non-null rows ({n_valid/len(panel)*100:.1f}%)")

print(f"\nRows with valid {PRIMARY_TARGET}: {panel[PRIMARY_TARGET].notna().sum()}")

poverty_3: 1999 non-null rows (100.0%)
poverty_8_30: 1865 non-null rows (93.3%)
poverty_10: 1865 non-null rows (93.3%)

Rows with valid poverty_3: 1999


## Train / Test split

80/20 random split, stratified by `country_code` so each country appears in both sets when possible.  
We use `random_state=42` for reproducibility.

In [6]:
# Use GroupShuffleSplit to keep country groups partially in both splits
# Simpler approach: stratify by country_code using train_test_split
# Since some countries have very few rows, we group rare countries together

# Count rows per country
country_counts = panel["country_code"].value_counts()
print(f"Countries with 1 row: {(country_counts == 1).sum()}")
print(f"Countries with 2+ rows: {(country_counts >= 2).sum()}")

# For stratification: countries with <2 rows can't be stratified,
# so we create a stratification key that groups rare countries together
rare_countries = set(country_counts[country_counts < 2].index)
panel["strat_key"] = panel["country_code"].apply(
    lambda x: "_RARE_" if x in rare_countries else x
)

train, test = train_test_split(
    panel,
    test_size=0.2,
    random_state=42,
    stratify=panel["strat_key"]
)

# Drop the helper column
train = train.drop(columns=["strat_key"]).reset_index(drop=True)
test = test.drop(columns=["strat_key"]).reset_index(drop=True)

print(f"\nTrain: {train.shape}")
print(f"Test:  {test.shape}")
print(f"Train countries: {train['country_code'].nunique()}")
print(f"Test countries:  {test['country_code'].nunique()}")

Countries with 1 row: 11
Countries with 2+ rows: 159

Train: (1599, 14)
Test:  (400, 14)
Train countries: 168
Test countries:  144


### Verify the split looks reasonable

In [7]:
print("=== Train set ===")
print(f"Year range: {train['year'].min()}–{train['year'].max()}")
print(f"{PRIMARY_TARGET} — mean: {train[PRIMARY_TARGET].mean():.2f}, "
      f"median: {train[PRIMARY_TARGET].median():.2f}, "
      f"non-null: {train[PRIMARY_TARGET].notna().sum()}")

print(f"\n=== Test set ===")
print(f"Year range: {test['year'].min()}–{test['year'].max()}")
print(f"{PRIMARY_TARGET} — mean: {test[PRIMARY_TARGET].mean():.2f}, "
      f"median: {test[PRIMARY_TARGET].median():.2f}, "
      f"non-null: {test[PRIMARY_TARGET].notna().sum()}")

print(f"\n=== Feature stats (train) ===")
print(train[FEATURES].describe().round(3))

=== Train set ===
Year range: 1996–2023
poverty_3 — mean: 10.56, median: 1.60, non-null: 1599

=== Test set ===
Year range: 1996–2023
poverty_3 — mean: 9.67, median: 1.30, non-null: 400

=== Feature stats (train) ===
       gdp_per_capita       hdi  control_of_corruption  \
count        1599.000  1599.000               1599.000   
mean        17200.993     0.759                  0.194   
std         22203.900     0.138                  1.035   
min           123.026     0.292                 -1.759   
25%          2711.777     0.686                 -0.620   
50%          6641.052     0.774                 -0.119   
75%         23953.396     0.874                  0.982   
max        141818.963     0.975                  2.459   

       employment_agriculture      gini  
count                1599.000  1599.000  
mean                   20.057    37.183  
std                    19.295     8.449  
min                     0.606    23.200  
25%                     4.128    30.850  
50%     

## Save train and test sets

In [8]:
train.to_csv(DATA_PROCESSED / "train.csv", index=False)
test.to_csv(DATA_PROCESSED / "test.csv", index=False)

print(f"Saved: {DATA_PROCESSED / 'train.csv'} — {train.shape}")
print(f"Saved: {DATA_PROCESSED / 'test.csv'}  — {test.shape}")

Saved: ../data/processed/train.csv — (1599, 14)
Saved: ../data/processed/test.csv  — (400, 14)


## Note on scaling

Tree-based models (XGBoost, LightGBM, Random Forest) do **not** need feature scaling.  
For Ridge regression and MLP, we will apply `StandardScaler` in the training notebook,  
fit on training data only to avoid data leakage.

---
**Outputs produced:**
- `data/processed/train.csv` — 80% of data, all columns preserved
- `data/processed/test.csv` — 20% of data, all columns preserved

Both files contain features, all 3 poverty targets, metadata (country, year), and `imputation_pct`.

> `poverty_4_20` excluded — source file contains poverty gap, not headcount ratio.